In [4]:
import numpy as np
from pathlib import Path
import sys
import os
import xarray as xr

In [5]:
root_dir = Path(os.getcwd()).parent.parent.parent
data_dir = root_dir / 'data' / 'ca_imaging'

# Add the model directory to Python path
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

Load original data

In [6]:
filenames = [data_dir / "cells_train.npz",
              data_dir / "cells_test.npz"]

cells_train = np.load(data_dir / 'cells_train.npz', allow_pickle=True)['arr_0'].item()
cells_test = np.load(data_dir / 'cells_test.npz', allow_pickle=True)['arr_0'].item()

imgs_train = np.array(cells_train['imgs']).transpose(0, 3, 1, 2)
masks_train = np.array(cells_train['masks'])
imgs_test = np.array(cells_test['imgs']).transpose(0, 3, 1, 2)
masks_test = np.array(cells_test['masks'])

Concatenate training and test data

In [ ]:
imgs = np.concatenate([imgs_train, imgs_test], axis = 0)
masks = np.concatenate([masks_train, masks_test], axis = 0)
# make cell present/no cell present labels for each pixel
labels = (masks > 0).astype(np.longlong)

Create xarray DataArrays for images and labels and put them together in xarray Dataset.

In [ ]:

imgs_xarr = xr.DataArray(
        data=imgs,
        dims=['image_nr', 'staining', 'height', 'width'],
        coords={'image_nr': ('image_nr', np.arange(imgs.shape[0])),
                'staining' : ('staining', ['cytoplasm', 'nuclear'])},
        name='ca_images',
        attrs={
            'description': 'Calcium imaging data with two kinds of staining',
            'notes': 'Cytoplasm means whole cell stained, nuclear means only nucleus of cells stained.'
        }
    )

labels_xarr = xr.DataArray(
        data=labels,
        dims=['image_nr', 'height', 'width'],
        coords={'image_nr': ('image_nr', np.arange(imgs.shape[0]))},
        name='cell_labels',
        attrs={
            'description': 'Cell labels',
            'notes': '1 means there is a cell at pixel, 0 means there is no cell at pixel.'
        }
    )

ds = xr.Dataset({
    'ca_image_data': imgs_xarr,
    'cell_labels': labels_xarr
})
ds

<xarray.Dataset> Size: 178MB
Dimensions:        (image_nr: 91, staining: 2, height: 383, width: 512)
Coordinates:
  * image_nr       (image_nr) int64 728B 0 1 2 3 4 5 6 ... 84 85 86 87 88 89 90
  * staining       (staining) <U9 72B 'cytoplasm' 'nuclear'
Dimensions without coordinates: height, width
Data variables:
    ca_image_data  (image_nr, staining, height, width) uint8 36MB 0 0 0 ... 8 8
    cell_labels    (image_nr, height, width) int64 143MB 0 0 0 0 0 ... 0 0 0 0 0

Write dataset to file

In [ ]:
ds.to_netcdf(data_dir / 'cells_data_all.nc')

Check saved file

In [20]:
dataset_check = xr.load_dataset(data_dir / 'cells_data_all.nc')

dataset_check

<xarray.Dataset> Size: 178MB
Dimensions:        (image_nr: 91, staining: 2, height: 383, width: 512)
Coordinates:
  * image_nr       (image_nr) int64 728B 0 1 2 3 4 5 6 ... 84 85 86 87 88 89 90
  * staining       (staining) <U9 72B 'cytoplasm' 'nuclear'
Dimensions without coordinates: height, width
Data variables:
    ca_image_data  (image_nr, staining, height, width) uint8 36MB 0 0 0 ... 8 8
    cell_labels    (image_nr, height, width) int64 143MB 0 0 0 0 0 ... 0 0 0 0 0